# UAW-AFS Frequency Points Contract

This notebook validates the planned job/worker contract for UAW-AFS frequency-point simulations without launching HFSS. It uses fake setup objects to show the two intended paths:

- no `frequency_points_ghz`: keep the template sweep, such as `LINC 3.5GHz 8.7GHz 401`;
- with `frequency_points_ghz`: replace the job sweep with a PyAEDT single-point sweep over exactly those GHz points.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'src').exists():
    REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

from worker.schemas import HfssSweepConfig, JobRequest
from worker.setup_designer import HfssSetupDesigner

REPO_ROOT

## Current Template Sweep

The existing template contract is still the default. For BCLF, the worker template typically defines a full interpolating sweep such as `LINC 3.5GHz 8.7GHz 401`.

In [ ]:
template_sweep = HfssSweepConfig(
    sweep_type='interpolating',
    sweep_data='LINC 3.5GHz 8.7GHz 401',
)

job_default = JobRequest(
    job_id='uaw_afs_contract_default',
    template_name='bclf_template_v1',
    setup_name='Setup1',
    sweep_name='Sweep1',
    variables={'L': '7.8mm', 'L1': '8.1mm', 'L2': '7.9mm', 'W': '1.0mm'},
    output_filename='uaw_afs_contract_default.s2p',
    metrics_to_extract=['max_s11_db', 'min_s21_db'],
    submit_time='2026-06-01T00:00:00Z',
)

job_default.frequency_points_ghz is None, template_sweep

## Fake HFSS Objects

The fake objects record which PyAEDT-style method would be called by the worker. They do not solve or export anything.

In [ ]:
class FakeSweep:
    def __init__(self, name):
        self.name = name
        self.props = {}


class FakeSetup:
    def __init__(self, name='Setup1'):
        self.name = name
        self.sweeps = []

    def get_sweep(self, name=None):
        if name is None:
            return self.sweeps[0] if self.sweeps else False
        return next((sweep for sweep in self.sweeps if sweep.name == name), False)

    def delete_sweep(self, name):
        for idx, sweep in enumerate(self.sweeps):
            if sweep.name == name:
                del self.sweeps[idx]
                return True
        return False

    def _add(self, name, kind):
        sweep = FakeSweep(name)
        sweep.props['kind'] = kind
        self.sweeps.append(sweep)
        return sweep

    def create_frequency_sweep(self, start_frequency=None, stop_frequency=None, num_of_freq_points=None, name=None, sweep_type='Discrete'):
        sweep = self._add(name or 'Sweep', 'frequency_sweep')
        sweep.props.update({
            'sweep_type': sweep_type,
            'start_frequency': start_frequency,
            'stop_frequency': stop_frequency,
            'num_of_freq_points': num_of_freq_points,
        })
        return sweep

    def create_linear_step_sweep(self, start_frequency=None, stop_frequency=None, step_size=None, name=None, sweep_type='Discrete'):
        sweep = self._add(name or 'Sweep', 'linear_step_sweep')
        sweep.props.update({
            'sweep_type': sweep_type,
            'start_frequency': start_frequency,
            'stop_frequency': stop_frequency,
            'step_size': step_size,
        })
        return sweep

    def create_single_point_sweep(self, name=None, unit='GHz', freq=None, save_single_field=False, save_fields=False):
        sweep = self._add(name or 'Sweep', 'single_point_sweep')
        sweep.props.update({
            'unit': unit,
            'freq': list(freq or []),
            'save_single_field': save_single_field,
            'save_fields': save_fields,
        })
        return sweep


class FakeHfss:
    def __init__(self):
        self.setups = []

    def create_setup(self, name):
        setup = FakeSetup(name)
        self.setups.append(setup)
        return setup

## Default Path: Keep Template Sweep

In [ ]:
hfss = FakeHfss()
resolved = HfssSetupDesigner().ensure_setup(
    hfss,
    setup_name='Setup1',
    sweep_name='Sweep1',
    sweep_config=template_sweep,
)

sweep = hfss.setups[0].sweeps[0]
assert resolved.setup_name == 'Setup1'
assert sweep.props['kind'] == 'frequency_sweep'
assert sweep.props['start_frequency'] == '3.5GHz'
assert sweep.props['stop_frequency'] == '8.7GHz'
assert sweep.props['num_of_freq_points'] == 401
sweep.props

## UAW-AFS Path: Override With Specific Frequency Points

`JobRequest` normalizes the optional GHz list by sorting and removing duplicates. The worker then creates a single-point sweep over those values.

In [ ]:
job_uaw = JobRequest(
    job_id='uaw_afs_contract_points',
    template_name='bclf_template_v1',
    setup_name='Setup1',
    sweep_name='Sweep1',
    frequency_points_ghz=[5.4, 3.5, 4.8, 5.4, 8.7],
    variables={'L': '7.8mm', 'L1': '8.1mm', 'L2': '7.9mm', 'W': '1.0mm'},
    output_filename='uaw_afs_contract_points.s2p',
    metrics_to_extract=['max_s11_db', 'min_s21_db'],
    submit_time='2026-06-01T00:00:00Z',
)

hfss = FakeHfss()
HfssSetupDesigner().ensure_setup(
    hfss,
    setup_name='Setup1',
    sweep_name='Sweep1',
    sweep_config=template_sweep,
    frequency_points_ghz=job_uaw.frequency_points_ghz,
)

sweep = hfss.setups[0].sweeps[0]
assert job_uaw.frequency_points_ghz == [3.5, 4.8, 5.4, 8.7]
assert sweep.props['kind'] == 'single_point_sweep'
assert sweep.props['unit'] == 'GHz'
assert sweep.props['freq'] == [3.5, 4.8, 5.4, 8.7]
assert sweep.props['save_single_field'] is False
assert sweep.props['save_fields'] is False
sweep.props

## Takeaway

The fake contract validates the intended data flow. A later real-HFSS smoke should submit one tiny `frequency_points_ghz` job and check that the exported Touchstone file contains exactly the requested frequency rows.

## Server-Side UAW-AFS Worker Job Submission

This section assumes the notebook runs on the server side. It selects one existing BCLF/UAW sample, draws 20 random frequency points from the same source frequency grid, and prepares a worker job with `frequency_points_ghz`. The notebook does not call PyAEDT directly; the Windows worker claims `queue/pending/<job_id>.json`, runs HFSS, uploads results, and the optional watcher can archive artifacts into `ext/afs`.

In [ ]:
import json
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import skrf as rf

from common.io_utils import read_json
from common.job_types import SimulationJob
from server.queue_manager import QueueManager
from worker.schemas import ArtifactOptions, JobRequest

EXT_ROOT = REPO_ROOT / 'data' / 'bandpass_coupled_line_filter_export' / 'ext'
AFS_ROOT = EXT_ROOT / 'afs'
FALLBACK_RESULTS_ROOT = REPO_ROOT / 'queue' / 'results'
QUEUE_ROOT = REPO_ROOT / 'queue'
N_FREQ_POINTS = 20
RANDOM_SEED = 0
TEMPLATE_NAME = 'bclf_template_v1'
DESIGN_NAME = 'HFSSDesign1'
SETUP_NAME = 'Setup1'
SWEEP_NAME = 'Sweep1'
VARIABLE_ORDER = ['L', 'L1', 'L2', 'W', 'W1', 'W2', 'S1', 'S2']
BCLF_TOUCHSTONE_PREFIX = 'Bandpass_Coupled_Line_Filter_export_Bandpass_Coupled_Line_Filter'
EXPORT_CSV = False
RUN_SUBMIT_WORKER_JOB = False
WAIT_FOR_WORKER_RESULT = False
WORKER_TIMEOUT_SEC = 7200
WORKER_POLL_INTERVAL_SEC = 10

CONFIG_SUMMARY = {
    'ext_root': str(EXT_ROOT),
    'afs_root': str(AFS_ROOT),
    'fallback_results_root': str(FALLBACK_RESULTS_ROOT),
    'queue_root': str(QUEUE_ROOT),
    'n_freq_points': N_FREQ_POINTS,
    'random_seed': RANDOM_SEED,
    'template_name': TEMPLATE_NAME,
    'design_name': DESIGN_NAME,
    'setup_name': SETUP_NAME,
    'sweep_name': SWEEP_NAME,
    'run_submit_worker_job': RUN_SUBMIT_WORKER_JOB,
    'wait_for_worker_result': WAIT_FOR_WORKER_RESULT,
}
CONFIG_SUMMARY

In [ ]:
def _is_touchstone(path: Path) -> bool:
    suffix = path.suffix.lower()
    return suffix.startswith('.s') and suffix.endswith('p')


def _decode_touchstone_value(encoded: str) -> float:
    text = encoded.strip()
    sign = -1.0 if text.startswith('m') else 1.0
    if text.startswith('m'):
        text = text[1:]
    return sign * float(text.replace('d', '.'))


def parse_bclf_variables_from_filename(path: Path, variable_order=VARIABLE_ORDER) -> dict[str, float]:
    parts = path.stem.split('_')
    parsed: dict[str, float] = {}
    expected = set(variable_order)
    idx = 0
    while idx < len(parts) - 1:
        key = parts[idx]
        if key in expected and key not in parsed:
            parsed[key] = _decode_touchstone_value(parts[idx + 1])
            idx += 2
        else:
            idx += 1
    missing = [name for name in variable_order if name not in parsed]
    if missing:
        raise ValueError(f'Could not parse variables {missing} from filename: {path.name}')
    return {name: float(parsed[name]) for name in variable_order}


def _variables_from_index(source_path: Path, ext_root: Path, variable_order=VARIABLE_ORDER) -> dict[str, float] | None:
    index_path = ext_root / 'index.json'
    if not index_path.exists():
        return None
    with index_path.open('r', encoding='utf-8') as fh:
        payload = json.load(fh)
    records = payload.get('records', {}) if isinstance(payload, dict) else {}
    if not isinstance(records, dict):
        return None
    for record in records.values():
        if not isinstance(record, dict):
            continue
        filename = record.get('filename')
        if filename is None or Path(str(filename)).name != source_path.name:
            continue
        variables = record.get('variables')
        if isinstance(variables, dict) and all(name in variables for name in variable_order):
            return {name: float(variables[name]) for name in variable_order}
    return None


def discover_source_touchstones(ext_root: Path, fallback_results_root: Path) -> tuple[list[Path], str]:
    if ext_root.exists():
        ext_candidates = sorted(
            path for path in ext_root.glob('*.s*p')
            if path.is_file() and _is_touchstone(path) and path.parent != (ext_root / 'afs')
        )
        if ext_candidates:
            return ext_candidates, 'ext'
    fallback_candidates = sorted(
        path for path in fallback_results_root.glob(f'{BCLF_TOUCHSTONE_PREFIX}*.s*p')
        if path.is_file() and _is_touchstone(path)
    )
    if fallback_candidates:
        return fallback_candidates, 'queue_results_fallback'
    raise FileNotFoundError(
        f'No source BCLF Touchstone files found in {ext_root} or {fallback_results_root}'
    )


def choose_source_sample(rng: np.random.Generator) -> tuple[Path, str, dict[str, float]]:
    candidates, source_kind = discover_source_touchstones(EXT_ROOT, FALLBACK_RESULTS_ROOT)
    selected = candidates[int(rng.integers(0, len(candidates)))]
    variables = _variables_from_index(selected, EXT_ROOT) if source_kind == 'ext' else None
    if variables is None:
        variables = parse_bclf_variables_from_filename(selected)
    return selected, source_kind, variables


def sample_frequency_points_ghz(source_touchstone: Path, n_points: int, rng: np.random.Generator) -> tuple[np.ndarray, np.ndarray]:
    network = rf.Network(str(source_touchstone))
    freq_hz = np.asarray(network.f, dtype=float).reshape(-1)
    if n_points > len(freq_hz):
        raise ValueError(f'n_points={n_points} exceeds available source frequencies={len(freq_hz)}')
    indices = np.sort(rng.choice(np.arange(len(freq_hz)), size=int(n_points), replace=False))
    sampled_hz = freq_hz[indices]
    sampled_ghz = sampled_hz / 1e9
    return sampled_hz, sampled_ghz


def make_afs_job_id(source_path: Path, seed: int, n_points: int) -> str:
    timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    return f'uaw_afs_{source_path.stem[:48]}_seed{seed}_n{n_points}_{timestamp}'


def make_output_filename(job_id: str) -> str:
    return f'{job_id}.s2p'


def build_worker_job_payload(job_id: str, variables: dict[str, float], sampled_freq_ghz: np.ndarray) -> JobRequest:
    return JobRequest(
        job_id=job_id,
        template_name=TEMPLATE_NAME,
        design_name=DESIGN_NAME,
        setup_name=SETUP_NAME,
        sweep_name=SWEEP_NAME,
        frequency_points_ghz=[float(x) for x in sampled_freq_ghz],
        variables={name: f'{variables[name]:g}mm' for name in VARIABLE_ORDER},
        output_filename=make_output_filename(job_id),
        metrics_to_extract=['max_s11_db', 'min_s21_db'],
        artifact_options=ArtifactOptions(export_touchstone=True, export_csv=EXPORT_CSV),
        submit_time=datetime.now(timezone.utc),
        retry_count=0,
    )

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
selected_touchstone, source_kind, selected_variables = choose_source_sample(rng)
sampled_freq_hz, sampled_freq_ghz = sample_frequency_points_ghz(selected_touchstone, N_FREQ_POINTS, rng)
network = rf.Network(str(selected_touchstone))
dry_run_job_id = make_afs_job_id(selected_touchstone, RANDOM_SEED, N_FREQ_POINTS)
dry_run_job = build_worker_job_payload(dry_run_job_id, selected_variables, sampled_freq_ghz)

DRY_RUN_SELECTION = {
    'source_kind': source_kind,
    'source_touchstone': str(selected_touchstone),
    'parsed_variables': selected_variables,
    'source_freq_count': int(len(network.f)),
    'source_freq_min_ghz': float(network.f[0] / 1e9),
    'source_freq_max_ghz': float(network.f[-1] / 1e9),
    'sampled_freq_count': int(len(sampled_freq_ghz)),
    'sampled_freq_ghz': [float(x) for x in sampled_freq_ghz],
    'worker_job_preview': dry_run_job.model_dump(mode='json'),
}

assert len(sampled_freq_ghz) == N_FREQ_POINTS
assert all(name in selected_variables for name in VARIABLE_ORDER)
assert dry_run_job.frequency_points_ghz == sorted(set(float(x) for x in sampled_freq_ghz))
DRY_RUN_SELECTION

## Submit To Worker Queue

Set `RUN_SUBMIT_WORKER_JOB = True` to write one canonical worker job JSON into `queue/pending`. If `WAIT_FOR_WORKER_RESULT` is also true, the notebook polls `queue/done` and `queue/failed`; when successful, it copies the worker Touchstone artifact and writes `.metadata.json` plus `.npz` under `ext/afs`.

In [ ]:
def _simulation_job_from_request(job: JobRequest) -> SimulationJob:
    artifact_options = job.artifact_options.model_dump(mode='json')
    return SimulationJob(
        job_id=job.job_id,
        template_name=job.template_name,
        variables=dict(job.variables),
        output_filename=job.output_filename,
        metrics_to_extract=list(job.metrics_to_extract),
        submit_time=job.submit_time.isoformat(),
        design_name=job.design_name,
        setup_name=job.setup_name,
        sweep_name=job.sweep_name,
        frequency_points_ghz=list(job.frequency_points_ghz or []),
        artifact_options=artifact_options,
        retry_count=job.retry_count,
    )


def submit_worker_job(job: JobRequest, source_touchstone: Path, source_kind: str, variables: dict[str, float], sampled_hz: np.ndarray) -> dict:
    AFS_ROOT.mkdir(parents=True, exist_ok=True)
    queue = QueueManager(QUEUE_ROOT)
    for subdir in ['pending', 'running', 'done', 'failed', 'results']:
        (QUEUE_ROOT / subdir).mkdir(parents=True, exist_ok=True)
    pending_path = queue.enqueue(_simulation_job_from_request(job))
    submission = {
        'status': 'submitted',
        'job_id': job.job_id,
        'pending_path': str(pending_path),
        'queue_root': str(QUEUE_ROOT),
        'source_kind': source_kind,
        'source_touchstone': str(source_touchstone),
        'variables': variables,
        'frequency_points_ghz': list(job.frequency_points_ghz or []),
        'frequency_points_hz': [float(x) for x in sampled_hz],
        'worker_job': job.model_dump(mode='json'),
        'submitted_at_utc': datetime.now(timezone.utc).isoformat(),
    }
    submission_path = AFS_ROOT / f'{job.job_id}.submission.json'
    submission_path.write_text(json.dumps(submission, indent=2, ensure_ascii=True), encoding='utf-8')
    submission['submission_path'] = str(submission_path)
    return submission


def wait_for_worker_result(job_id: str, timeout_sec: int = WORKER_TIMEOUT_SEC, poll_interval_sec: int = WORKER_POLL_INTERVAL_SEC) -> tuple[Path, dict]:
    deadline = time.monotonic() + float(timeout_sec)
    done_path = QUEUE_ROOT / 'done' / f'{job_id}.result.json'
    failed_path = QUEUE_ROOT / 'failed' / f'{job_id}.result.json'
    while True:
        if done_path.exists():
            return done_path, read_json(done_path)
        if failed_path.exists():
            return failed_path, read_json(failed_path)
        if time.monotonic() >= deadline:
            raise TimeoutError(f'Timed out waiting for worker result: {job_id}')
        time.sleep(float(poll_interval_sec))


def archive_worker_result_to_afs(job: JobRequest, result_path: Path, result_payload: dict, submission: dict) -> dict:
    AFS_ROOT.mkdir(parents=True, exist_ok=True)
    artifacts = result_payload.get('artifacts') if isinstance(result_payload, dict) else None
    if not isinstance(artifacts, dict):
        artifacts = {}
    touchstone_rel = artifacts.get('touchstone')
    if not isinstance(touchstone_rel, str) or not touchstone_rel:
        raise ValueError(f'Worker result has no touchstone artifact: {result_path}')
    worker_touchstone = QUEUE_ROOT / touchstone_rel
    if not worker_touchstone.exists():
        raise FileNotFoundError(f'Worker touchstone artifact not found: {worker_touchstone}')

    afs_touchstone = AFS_ROOT / Path(job.output_filename).name
    shutil.copy2(worker_touchstone, afs_touchstone)
    csv_rel = artifacts.get('csv')
    afs_csv = None
    if isinstance(csv_rel, str) and csv_rel:
        worker_csv = QUEUE_ROOT / csv_rel
        if worker_csv.exists():
            afs_csv = AFS_ROOT / worker_csv.name
            shutil.copy2(worker_csv, afs_csv)

    network = rf.Network(str(afs_touchstone))
    npz_path = AFS_ROOT / f'{job.job_id}.npz'
    np.savez_compressed(
        npz_path,
        freq_hz=np.asarray(network.f, dtype=float),
        s_params=np.asarray(network.s, dtype=complex),
        variables=np.asarray([submission['variables'][name] for name in VARIABLE_ORDER], dtype=float),
        variable_order=np.asarray(VARIABLE_ORDER),
        source_touchstone=submission['source_touchstone'],
    )
    metadata = {
        **submission,
        'status': str(result_payload.get('status', result_path.parent.name)),
        'result_path': str(result_path),
        'result_payload': result_payload,
        'afs_touchstone': str(afs_touchstone),
        'afs_csv': str(afs_csv) if afs_csv is not None else None,
        'npz_path': str(npz_path),
        'archived_at_utc': datetime.now(timezone.utc).isoformat(),
        'resim_touchstone_freq_count': int(len(network.f)),
    }
    metadata_path = AFS_ROOT / f'{job.job_id}.metadata.json'
    metadata_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=True), encoding='utf-8')
    if metadata['resim_touchstone_freq_count'] != N_FREQ_POINTS:
        raise AssertionError(
            f"Expected {N_FREQ_POINTS} exported frequency rows, got {metadata['resim_touchstone_freq_count']}"
        )
    metadata['metadata_path'] = str(metadata_path)
    return metadata


def submit_uaw_afs_worker_job() -> dict:
    rng = np.random.default_rng(RANDOM_SEED)
    source_touchstone, source_kind, variables = choose_source_sample(rng)
    sampled_hz, sampled_ghz = sample_frequency_points_ghz(source_touchstone, N_FREQ_POINTS, rng)
    job_id = make_afs_job_id(source_touchstone, RANDOM_SEED, N_FREQ_POINTS)
    job = build_worker_job_payload(job_id, variables, sampled_ghz)
    submission = submit_worker_job(job, source_touchstone, source_kind, variables, sampled_hz)
    if not WAIT_FOR_WORKER_RESULT:
        return submission
    result_path, result_payload = wait_for_worker_result(job.job_id)
    if result_payload.get('status') != 'done':
        return {**submission, 'status': 'failed', 'result_path': str(result_path), 'result_payload': result_payload}
    return archive_worker_result_to_afs(job, result_path, result_payload, submission)


if RUN_SUBMIT_WORKER_JOB:
    WORKER_SUBMISSION_RESULT = submit_uaw_afs_worker_job()
else:
    WORKER_SUBMISSION_RESULT = {
        'status': 'skipped',
        'reason': 'Set RUN_SUBMIT_WORKER_JOB = True to write queue/pending job JSON for the worker.',
        'dry_run_source_touchstone': DRY_RUN_SELECTION['source_touchstone'],
        'dry_run_sampled_freq_count': DRY_RUN_SELECTION['sampled_freq_count'],
        'worker_job_preview': DRY_RUN_SELECTION['worker_job_preview'],
    }

WORKER_SUBMISSION_RESULT